# Step 5 — Refine schema

> Drop redundant or low-value columns (intermediate IDs, raw API blobs) so the subsequent filtering steps work on a slim, audit-friendly schema.

**Paper section:** Step 1 — Repository selection and workflow extraction (offline preparation).

## Inputs
| Path | Format | Source |
|---|---|---|
| `data/4_final_dataset.csv` | CSV | Step 4 |

## Outputs
| Path | Format | Consumer |
|---|---|---|
| `data/5_dataset.csv` | CSV (slim schema) | Step 6 (bot classification) |
| `data/step5_report.json` | JSON | Step 6 |



In [1]:
# --- Notebook bootstrap ---
# Idempotent: chdir to repo root + reset sys.argv for argparse.
# Works in Jupyter/JupyterLab/Antigravity/VS Code/plain python on Linux+macOS+Windows.
from dataset_utils.notebook_bootstrap import setup

setup()

print("Step 5 — starting: refine schema...", flush=True)
"""Step 5: refine columns of the merged dataset.

Removes columns that are:
- PII or high cardinality noise (author_email, committer_name, committer_email)
- Historical metadata irrelevant after deduplication (previous_file_path, previous_file_hash)
- Constant/already filtered (probably_workflow, valid_workflow, valid_yaml)
- Repository-level counts unrelated to workflows (repo_branches, repo_releases)

Output: data/5_dataset.csv (final clean dataset)
"""

import argparse
import json
from dataclasses import asdict, dataclass
from pathlib import Path

import polars as pl

COLUMNS_TO_REMOVE = {
    "author_email",
    "committer_name",
    "committer_email",
    "previous_file_path",
    "previous_file_hash",
    "probably_workflow",
    "valid_workflow",
    "valid_yaml",
    "repo_branches",
    "repo_releases",
}

@dataclass
class RefineStats:
    total_rows: int = 0
    rows_written: int = 0
    columns_removed: int = 0
    columns_kept: int = 0

def refine_columns(input_path: Path, output_path: Path) -> RefineStats:
    stats = RefineStats()

    from dataset_utils.integrity import assert_input_matches_report

    input_sha = assert_input_matches_report(
        input_path=input_path,
        prior_report_path=Path("data/step4_report.json"),
        label="step 5 input",
    )

    print("  reading input CSV (may take a moment for large files)...", flush=True)
    df = pl.read_csv(input_path, encoding="utf-8-sig")
    stats.total_rows = len(df)

    original_columns = df.columns
    output_columns = [col for col in original_columns if col not in COLUMNS_TO_REMOVE]

    stats.columns_removed = len(original_columns) - len(output_columns)
    stats.columns_kept = len(output_columns)

    df = df.select(output_columns)
    stats.rows_written = len(df)

    output_path.parent.mkdir(parents=True, exist_ok=True)
    print("  writing output...", flush=True)
    df.write_csv(output_path, include_header=True)

    from dataset_utils.hashing import compute_file_hash
    report = {
        **asdict(stats),
        "input_sha256": input_sha,
        "output_file": str(output_path),
        "output_sha256": compute_file_hash(output_path),
    }
    report_path = output_path.parent / "step5_report.json"
    with report_path.open("w") as f:
        json.dump(report, f, indent=2)

    return stats

def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Refine columns of merged dataset, removing irrelevant fields."
    )
    parser.add_argument(
        "--input",
        type=Path,
        default=Path("data/4_final_dataset.csv"),
        help="Input merged CSV file path.",
    )
    parser.add_argument(
        "--output",
        type=Path,
        default=Path("data/5_dataset.csv"),
        help="Output refined CSV file path.",
    )
    return parser.parse_args()

def main() -> None:
    args = parse_args()
    stats = refine_columns(args.input, args.output)

    print("Column refinement finished.")
    print(f"Input: {args.input}")
    print(f"Output: {args.output}")
    print("-" * 40)
    print(f"Total rows read: {stats.total_rows}")
    print(f"Rows written: {stats.rows_written}")
    print("-" * 40)
    print(f"Columns removed: {stats.columns_removed}")
    print(f"Columns kept: {stats.columns_kept}")

main()


Step 5 — starting: refine schema...
  reading input CSV (may take a moment for large files)...
  writing output...
Column refinement finished.
Input: data/4_final_dataset.csv
Output: data/5_dataset.csv
----------------------------------------
Total rows read: 253921
Rows written: 253921
----------------------------------------
Columns removed: 10
Columns kept: 20


---

**Next step:** [`06_classify_bot_authors.ipynb`](./06_classify_bot_authors.ipynb)
